In [1]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time
import os

In [ ]:
# Initialize data storage for plotting
timestamps = []
predictions = []
conf_no_aeration = []
conf_aeration = []
SAVE_PATH = "/opt/local/water_prediction_plot.png"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

In [3]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "prediction_topic"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True
    )
    return consumer

In [4]:
# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            print(f"Received: {sensor_data}")

            # Update data storage
            timestamps.append(sensor_data['timestamp'])
            predictions.append(sensor_data['prediction'])
            conf_no_aeration.append(sensor_data['proba_no_areation'])
            conf_aeration.append(sensor_data['proba_areation'])

            # Keep only the last 100 entries for plotting
            if len(timestamps) > 100:
                timestamps.pop(0)
                predictions.pop(0)
                conf_no_aeration.pop(0)
                conf_aeration.pop(0)

            fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 12), sharex=True)

            # --- 3.3.1: Estado de aireación ---
            ax1.step(timestamps, predictions, where='post', color='blue', linewidth=2)
            ax1.set_title("Estado de Aireación por Timestamp (1=Sí, 0=No)")
            ax1.set_ylabel("Estado")
            ax1.grid(True, alpha=0.3)

            # --- 3.3.2: Confianza Clase Aireación (cuando es escogida) ---
            ax2.step(timestamps, conf_aeration, color='green', label='Confianza Aireación')
            ax2.set_title("Confianza: Clase Aireación (Solo cuando es la clase predicha)")
            ax2.set_ylabel("Probabilidad")
            ax2.set_ylim(0, 1.0) # La confianza siempre será > 0.5 si fue la elegida
            ax2.grid(True, alpha=0.3)

            # --- 3.3.3: Confianza Clase No Aireación (cuando es escogida) ---
            ax3.step(timestamps, conf_no_aeration, color='red', label='Confianza No Aireación')
            ax3.set_title("Confianza: Clase No Aireación (Solo cuando es la clase predicha)")
            ax3.set_ylabel("Probabilidad")
            ax3.set_xlabel("Timestamp")
            ax3.set_ylim(0, 1.0)
            ax3.grid(True, alpha=0.3)

            ax4.remove()

            # Formatear el eje X para que no se amontonen los timestamps
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(SAVE_PATH)
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

# 👂 Escuchamos

In [ ]:
consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality'.")

try:
    while True:
        update_plot(consumer)
        
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality'.
Received: {'timestamp': 1773918053, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773918054, 'prediction': 0, 'proba_no_areation': 1.0, 'proba_areation': 0.0}
Received: {'timestamp': 1773918055, 'prediction': 0, 'proba_no_areation': 1.0, 'proba_areation': 0.0}
Received: {'timestamp': 1773918056, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773918057, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773918058, 'prediction': 0, 'proba_no_areation': 1.0, 'proba_areation': 0.0}
Received: {'timestamp': 1773918059, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773918060, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773918061, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773918062, 'pr

Stopped consuming messages.
